# 🧪 POC — LLM Local no Google Colab (Ollama + Qwen3.5-4B)
## Rodando Qwen 3.5 4B via Ollama, sem API externa!

<a href="https://colab.research.google.com/github/luksamuk/guilda-ia/blob/main/notebooks/poc_ollama_qwen35_colab.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Objetivo:** Provar que é possível rodar um modelo pequeno (Qwen 3.5 — 4B parâmetros)
direto no Google Colab gratuito (T4 GPU) usando **Ollama**, sem precisar de API key externa.

**Por que Ollama em vez de llama.cpp?**
- Ollama expõe uma API REST HTTP (compatível com OpenAI!) em `localhost:11434`
- Os alunos usam `requests.post()` — exatamente como fariam com a Gemini API
- **Tool calling nativo** — o modelo pode chamar ferramentas
- **Thinking mode** nativo — raciocínio interno antes da resposta
- Setup mais enxuto: um binário + um comando `ollama pull`

**Status:** POC experimental — não é material de aula (ainda).

---


## 1. Verificar GPU

⚠️ **Importante:** Vá em `Runtime → Change runtime type` e selecione **T4 GPU** antes de continuar.


In [ ]:
# Verificar GPU disponível
!nvidia-smi


## 2. Instalar e Configurar o Ollama

O Ollama é um servidor de inference local que roda como daemon. Vamos:
1. Baixar e instalar o binário
2. Corrigir o `LD_LIBRARY_PATH` (workaround pro Colab reconhecer a GPU)
3. Iniciar o servidor em background


In [ ]:
# Instalar Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Workaround: Ollama não detecta a GPU no Colab sem ajustar o LD_LIBRARY_PATH
import os
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")

print("✅ Ollama instalado!")

## 3. Iniciar o Servidor Ollama

O Ollama roda como um servidor em background na porta 11434.
Vamos iniciá-lo e verificar que está respondendo.


In [ ]:
# Iniciar Ollama em background
import subprocess
import time
import requests

# Matar qualquer instância anterior (seguro rodar múltiplas vezes)
subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(1)

# Iniciar servidor em background
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env={**os.environ}
)

# Esperar o servidor ficar pronto (tentativas com timeout)
print("⏳ Aguardando Ollama iniciar...")
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code == 200:
            print("✅ Ollama rodando na porta 11434!")
            break
    except:
        time.sleep(1)
else:
    print("❌ Ollama não iniciou em 30s. Tente rodar a célula novamente.")

## 4. Baixar o Modelo Qwen 3.5 4B

Vamos usar `ollama pull` para baixar o **Qwen3.5-4B** (~3.4 GB, quantização Q4_K_M).

O download leva ~1-3 min dependendo da velocidade do Colab.


In [ ]:
# Baixar o modelo Qwen 3.5 4B
# --insecure é necessário no Colab por causa do proxy/environment
!ollama pull qwen3.5:4b

print("\n✅ Modelo qwen3.5:4b baixado!")

## 5. Verificar Modelo e Teste Rápido

Vamos confirmar que o modelo está disponível e fazer um teste simples via CLI.


In [ ]:
# Listar modelos disponíveis
!ollama list

print("\n---\n")
print("Teste rápido via CLI:")
!ollama run qwen3.5:4b "Diga 'Olá, Guilda de IA!' e nada mais." --nowidthify

## 6. Inferência via API REST (Python `requests`)

Aqui está o **ponto principal da POC**: o Ollama expõe uma API HTTP compatível com OpenAI.
Isso significa que o código Python é **idêntico** ao que usaríamos com a Gemini API ou OpenAI —
só muda a URL base de `https://generativelanguage.googleapis.com/...` pra `http://localhost:11434/...`.


In [ ]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/chat"

def chat(messages, model="qwen3.5:4b", stream=False):
    """Envia uma mensagem pro Ollama e retorna a resposta.

    Args:
        messages: lista de dicts no formato OpenAI [{"role": "...", "content": "..."}]
        model: nome do modelo no Ollama
        stream: se True, retorna generator de chunks

    Returns:
        dict com a resposta completa (ou generator se stream=True)
    """
    payload = {
        "model": model,
        "messages": messages,
        "stream": stream
    }

    response = requests.post(OLLAMA_URL, json=payload, timeout=120)

    if response.status_code != 200:
        raise Exception(f"Erro {response.status_code}: {response.text}")

    # Ollama retorna JSON não-stream ou NDJSON stream
    if stream:
        return response
    return response.json()

# Teste simples
resultado = chat([
    {"role": "system", "content": "Você é um assistente útil. Responda em português."},
    {"role": "user", "content": "Qual é a capital de Minas Gerais?"}
])

print(f"🤖 Resposta: {resultado['message']['content']}")
print(f"📊 Tokens: prompt={resultado.get('prompt_eval_count', '?')}, "
      f"completion={resultado.get('eval_count', '?')}")

## 7. Chat com Memória (Conversa Multi-turno)

A API do Ollama aceita o histórico completo de mensagens — igual à Gemini API.
Cada chamada envia toda a conversa, e o modelo mantém o contexto.


In [ ]:
# Chat com memória — acumulando o histórico
historico = [
    {"role": "system", "content": "Você é um assistente útil. Responda em português. Seja conciso."}
]

# Turno 1
historico.append({"role": "user", "content": "Meu nome é Lucas e eu ensino IA."})
r1 = chat(historico)
historico.append({"role": "assistant", "content": r1["message"]["content"]})
print(f"🤖: {r1['message']['content']}")

# Turno 2 — o modelo lembra do nome?
historico.append({"role": "user", "content": "Qual é o meu nome?"})
r2 = chat(historico)
historico.append({"role": "assistant", "content": r2["message"]["content"]})
print(f"🤖: {r2['message']['content']}")

## 8. Modo Thinking (Raciocínio Interno)

O Qwen 3.5 suporta **thinking mode** — o modelo raciocina internamente antes de responder.
Para ativar, basta incluir `/think` na mensagem ou usar o parâmetro `think: true` na API.


In [ ]:
# Thinking mode — o modelo raciocina antes de responder
resultado = chat([
    {"role": "system", "content": "Você é um assistente útil. Responda em português."},
    {"role": "user", "content": "Quantos números primos existem entre 1 e 20? Resolva passo a passo. /think"}
])

# Ollama retorna thinking separado quando disponível
msg = resultado["message"]
thinking = msg.get("thinking", "(sem thinking)")
content = msg.get("content", "")

print(f"💭 Thinking:\n{thinking[:500]}...")
print(f"\n🤖 Resposta: {content}")

## 9. Streaming — Resposta em Tempo Real

Streaming mostra o texto conforme é gerado, igual ao ChatGPT.
Basta passar `stream=True` e iterar sobre os chunks.


In [ ]:
# Streaming de resposta
print("🤖 ", end="")

response = chat(
    [
        {"role": "system", "content": "Você é um assistente útil. Responda em português."},
        {"role": "user", "content": "Explique o que é uma API em 3 frases."}
    ],
    stream=True
)

for line in response.iter_lines():
    if line:
        chunk = json.loads(line)
        if "message" in chunk and "content" in chunk["message"]:
            text = chunk["message"]["content"]
            print(text, end="", flush=True)

print()  # newline no final

## 10. API Compatível com OpenAI

O Ollama também expõe um endpoint compatível com a OpenAI API em `/v1/chat/completions`.
Isso significa que **qualquer código que usa a OpenAI SDK pode apontar pro Ollama local**
trocando só a `base_url` e a `api_key`.

Isso é **muito relevante pra aula**: o padrão é o mesmo, só muda onde o request vai.


In [ ]:
# Usando o endpoint OpenAI-compatible
# Mesmo formato de request que a OpenAI API!

OPENAI_URL = "http://localhost:11434/v1/chat/completions"

payload = {
    "model": "qwen3.5:4b",
    "messages": [
        {"role": "system", "content": "Você é um assistente útil. Responda em português."},
        {"role": "user", "content": "O que é Python em uma frase?"}
    ],
    "temperature": 0.7,
    "max_tokens": 64
}

response = requests.post(OPENAI_URL, json=payload, timeout=60)
data = response.json()

# Formato idêntico ao da OpenAI!
print(f"🤖 Resposta: {data['choices'][0]['message']['content']}")
print(f"📊 Model: {data['model']}")
print(f"📊 Usage: {data['usage']}")

## 11. Benchmark de Velocidade

Vamos medir quantos tokens por segundo o Qwen3.5-4B gera no Colab gratuito.


In [ ]:
import time

# Benchmark: gerar ~100 tokens e medir velocidade
prompt = "Conte uma história curta sobre um robô que aprende a cozinhar."

start = time.time()
resultado = chat([
    {"role": "user", "content": prompt}
])
elapsed = time.time() - start

eval_count = resultado.get("eval_count", 0)
prompt_count = resultado.get("prompt_eval_count", 0)
eval_duration = resultado.get("eval_duration", 0) / 1e9  # nanos → segundos

speed = eval_count / eval_duration if eval_duration > 0 else 0

print(f"⏱️ Tempo total: {elapsed:.1f}s")
print(f"📊 Tokens: prompt={prompt_count}, completion={eval_count}")
print(f"🚀 Velocidade: {speed:.1f} tokens/s")
print(f"\n📝 Resposta: {resultado['message']['content'][:200]}...")

## 12. 📋 Resultados da POC

### O que funciona ✅
- **Qwen3.5-4B roda no Colab gratuito** (T4 GPU, 16 GB VRAM)
- Modelo Q4_K_M ocupa ~3.4 GB de download, ~6 GB de VRAM
- Ollama setup é mais enxuto que `llama-cpp-python` (sem compilação CUDA)
- **API REST HTTP** — alunos usam `requests.post()`, mesmo padrão da Gemini API
- **API compatível com OpenAI** (`/v1/chat/completions`) — fácil migração de código
- **Tool calling nativo** — o modelo pode chamar ferramentas
- **Thinking mode** — raciocínio interno separado da resposta
- Streaming funciona e melhora a experiência

### Por que Ollama é melhor que llama.cpp pra aula? 🎓

| Aspecto | `llama-cpp-python` (POC antiga) | **Ollama** (POC nova) |
|---------|-------------------------------|---------------------|
| Setup | Compila CUDA (~3-5 min) | Binário + pull (~2-3 min) |
| API | Python-only (`Llama.create_chat_completion()`) | **REST HTTP** (qualquer linguagem) |
| Tool calling | Não nativo | **Nativo** |
| Thinking | Via `extra_body` | **Nativo** (`/think`) |
| Compatibilidade | Binding específico | **OpenAI-compatible** |
| Didática | Ensina binding Python | **Ensina padrão HTTP** (transferível pra qualquer API) |

### Limitações ⚠️
- **Setup demorado:** ~2-3 min por sessão Colab
- **Sem persistência:** modelo baixado a cada sessão
- **Rate limits do Colab:** sessões gratuitas têm limite de horas
- **Workaround GPU:** Ollama precisa do `LD_LIBRARY_PATH` ajustado no Colab

### Vabilidade para a Guilda
- **Aula 04 (Python + API):** O Ollama pode ser o backend local — alunos fazem `requests.post()` num servidor local, aprendendo HTTP na prática sem API key
- **Aula futura (Agentes):** Tool calling nativo permite demonstrar agentes com ferramentas
- **Recomendação:** Usar Ollama local como complemento à Gemini API, mostrando o padrão HTTP unificado

---
*POC experimental — Guilda de IA UFVJM 2026.1*
